# Python — Generators & List Comprehensions
**Day 1 — Python Module**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>Core Insight:</strong> Generators are <em>lazy</em> — they produce one value at a time instead
of building the whole result in memory. For data engineering with millions of rows, this is
the difference between crashing and streaming. A generator expression on 10 million items
uses ~200 bytes. A list comprehension uses ~80 MB.
</div>

### Why It Matters
In data pipelines, you rarely need all records in memory at once. You need to process them one at a time (or in chunks). Generators are Python's native streaming mechanism.

## List Comprehensions — Fast and Readable

```python
# Standard pattern: [expression for item in iterable if condition]
squares = [x**2 for x in range(10) if x % 2 == 0]
# → [0, 4, 16, 36, 64]

# Nested: flatten a 2D list
matrix = [[1, 2], [3, 4], [5, 6]]
flat = [num for row in matrix for num in row]
# → [1, 2, 3, 4, 5, 6]

# Dict comprehension
word_lengths = {word: len(word) for word in ["data", "engineer", "SQL"]}
# → {'data': 4, 'engineer': 8, 'SQL': 3}

# Set comprehension (deduplication)
unique_lengths = {len(word) for word in ["data", "engineer", "SQL", "cat"]}
# → {3, 4, 8}
```

**Rule:** Use list comprehension when you need the full list in memory at once (small data, or data you'll iterate multiple times). Use a generator when processing once in sequence.

In [ ]:
# Memory comparison — list vs generator
import sys

# List comprehension: builds the entire list in memory immediately
N = 100_000
squares_list = [x**2 for x in range(N)]
print(f"List ({N:,} items):      {sys.getsizeof(squares_list):>10,} bytes  ← all in memory")

# Generator expression: lazy, holds only its state (loop counter + function ref)
squares_gen = (x**2 for x in range(N))
print(f"Generator ({N:,} items): {sys.getsizeof(squares_gen):>10,} bytes  ← ~208 bytes always")

# Both produce identical results — generator just can't be rewound
total_gen  = sum(squares_gen)    # consumes the generator (exhausted after this)
total_list = sum(squares_list)   # list is reusable — can iterate again
print(f"Equal totals: {total_gen == total_list}")  # True

# Demonstrate: at 10M items, list = ~80 MB, generator stays at ~208 bytes
print("\nAt 10M items: list ~80 MB vs generator ~208 bytes")
print("At 1B items:  list would crash; generator still uses 208 bytes")

## Generator Functions — `yield`

A generator function uses `yield` instead of `return`. When called, it returns a generator object — an iterator that runs the function body up to the next `yield`, pauses, and resumes on the next call.

**Execution model:**
1. Call the function → get a generator object (nothing runs yet)
2. `next(gen)` → runs until `yield`, suspends, returns the yielded value
3. `next(gen)` → resumes from after `yield`, runs until next `yield`
4. When function returns (or falls off the end) → raises `StopIteration`

In [ ]:
# Generator function — process a CSV in chunks (io.StringIO for demo)
import io

SAMPLE_CSV = """server_id,metric,value
srv-01,cpu_pct,72.5
srv-01,cpu_pct,73.1
srv-02,cpu_pct,45.2
srv-02,cpu_pct,46.0
srv-03,cpu_pct,88.4
srv-03,cpu_pct,91.2
srv-04,cpu_pct,34.1
srv-04,cpu_pct,35.0
srv-05,cpu_pct,78.9
srv-05,cpu_pct,80.1
srv-06,cpu_pct,95.3
srv-06,cpu_pct,97.1"""

def read_csv_chunks(fileobj, chunk_size: int = 4):
    """Yield lists of lines in chunks. Constant memory regardless of file size."""
    next(fileobj)   # skip header
    chunk = []
    for line in fileobj:
        line = line.rstrip()
        if line:
            chunk.append(line)
        if len(chunk) == chunk_size:
            yield chunk        # pause, hand chunk to caller
            chunk = []         # reset — old chunk is GC'd
    if chunk:
        yield chunk            # yield the final partial chunk

# Use io.StringIO as the file — same interface as open()
f = io.StringIO(SAMPLE_CSV)
for i, chunk in enumerate(read_csv_chunks(f, chunk_size=4)):
    print(f"Chunk {i+1}: {len(chunk)} rows → {chunk}")

# Simple counter generator to show yield/next mechanics
def counter_gen(n):
    i = 0
    while i < n:
        yield i
        i += 1

gen = counter_gen(5)
print("\ncounter_gen demo:")
print(next(gen))   # 0
print(next(gen))   # 1
print(list(gen))   # [2, 3, 4] — consume the rest

In [ ]:
# Real generator pipeline — runs against in-memory sample data
# Each stage is lazy: data flows one record at a time

SAMPLE_METRICS = """server_id,metric,value
srv-01,cpu_pct,72.5
srv-02,cpu_pct,91.3
srv-03,cpu_pct,45.0
srv-04,cpu_pct,88.7
srv-05,cpu_pct,62.1
srv-06,cpu_pct,95.4
srv-07,mem_pct,70.2
srv-08,cpu_pct,55.0"""

SERVER_TIERS = {
    'srv-01': 'production', 'srv-02': 'production', 'srv-03': 'staging',
    'srv-04': 'production', 'srv-05': 'staging',    'srv-06': 'production',
    'srv-07': 'production', 'srv-08': 'dev',
}

def parse_line(line: str) -> dict:
    """Parse CSV line into metric dict."""
    parts = line.strip().split(',')
    return {'server_id': parts[0], 'metric': parts[1], 'value': float(parts[2])}

def filter_high_cpu(records, threshold=80.0):
    """Generator: yield only CPU records above threshold."""
    for rec in records:
        if rec['metric'] == 'cpu_pct' and rec['value'] > threshold:
            yield rec

def enrich_with_tier(records, tiers: dict):
    """Generator: add tier to each record."""
    for rec in records:
        rec['tier'] = tiers.get(rec['server_id'], 'unknown')
        yield rec

# ── Build and execute the pipeline ──────────────────────────────────────────
import io
lines   = io.StringIO(SAMPLE_METRICS)
next(lines)                                         # skip header
parsed  = (parse_line(l) for l in lines)            # lazy parse
high    = filter_high_cpu(parsed, threshold=80)     # lazy filter
enriched = enrich_with_tier(high, SERVER_TIERS)    # lazy enrich
results = list(enriched)                            # materialize here only

print(f"High-CPU servers (>80%): {len(results)} found")
for r in results:
    print(f"  {r['server_id']} | {r['value']}% CPU | tier={r['tier']}")
print("\nEach generator stage used O(1) memory — no intermediate lists built")

## itertools — Production-Grade Iteration

`itertools` provides lazy combinators for generators. Key tools for data engineering:

In [ ]:
import itertools

# chain: combine multiple iterables lazily (no intermediate list)
servers   = ["srv-01", "srv-02", "srv-03"]
databases = ["db-01", "db-02"]
all_resources = list(itertools.chain(servers, databases))
print("chain:", all_resources)   # ['srv-01', 'srv-02', 'srv-03', 'db-01', 'db-02']

# islice: take the first N items from a generator (safe — doesn't exhaust it)
def infinite_counter(start=0):
    while True:
        yield start
        start += 1

first_five = list(itertools.islice(infinite_counter(100), 5))
print("islice:", first_five)    # [100, 101, 102, 103, 104]

# groupby: group consecutive items (sort first! groupby is NOT a SQL GROUP BY)
from itertools import groupby

data = sorted([
    ("production", "srv-01"), ("production", "srv-02"),
    ("staging",    "srv-03"), ("dev",        "srv-04"),
], key=lambda x: x[0])

for env, items in groupby(data, key=lambda x: x[0]):
    servers_in_env = [s for _, s in items]
    print(f"  {env}: {servers_in_env}")

# batched (Python 3.12+): split an iterable into batches of N
# For older Python, use islice in a loop:
def batched(iterable, n):
    it = iter(iterable)
    while batch := list(itertools.islice(it, n)):
        yield batch

for batch in batched(range(10), 3):
    print("batch:", batch)  # [0,1,2], [3,4,5], [6,7,8], [9]

## Interview Q&A

**Q: What's the difference between a list comprehension and a generator expression?**
A: A list comprehension `[x for x in ...]` builds the full list in memory immediately — eager evaluation. A generator expression `(x for x in ...)` is lazy — yields one value at a time. Use generators for large data where you only need one pass.

**Q: When does `yield` pause execution?**
A: Exactly at the `yield` statement. The function's local state (all variables, the position in the code) is saved. On the next `next()` call, execution resumes from the line after `yield`.

**Q: How would you process a 10GB CSV without loading it into memory?**
A: Use a generator that reads and yields rows (or chunks) one at a time. `pandas.read_csv(filepath, chunksize=1000)` is built on this exact pattern — it returns an iterator of DataFrames, each with 1000 rows.

**Q: What is a generator's key limitation vs a list?**
A: Generators are single-pass and forward-only. Once an element is yielded and consumed, you can't go back. If you need to iterate the data multiple times (e.g., compute mean, then compute variance), materialize to a list first — or use two separate generators.

**Q: What is `itertools.groupby` and what's the gotcha?**
A: `groupby` groups consecutive equal elements. The gotcha: it only groups consecutive items, not all items with the same key (unlike SQL GROUP BY). You **must sort by the grouping key first** or you'll get multiple groups for the same key.

## The Citi Angle

**At Citi, generators were the right tool for telemetry pipelines.** APM data from 6,000 servers generated millions of metric records per collection cycle. Loading everything into memory was never an option.

**The chunk-processing pattern:**
```python
def process_telemetry_file(path: str, batch_size: int = 5000):
    """Process telemetry CSV in memory-safe batches."""
    for chunk in read_csv_chunks(path, batch_size):
        # Each chunk is a list of 5000 lines
        records = [parse_metric(line) for line in chunk]
        records = [r for r in records if r['cpu_pct'] > 70]  # filter
        if records:
            insert_to_db(records)   # batch insert, not row-by-row
        # chunk goes out of scope, memory freed immediately
```

**Interview line:** *"For capacity data processing at Citi — 6,000 servers, millions of rows per cycle — Python generators were essential. The chunk pattern kept memory constant while processing arbitrarily large files. Same mental model as Spark's lazy execution: don't materialize until you have to."*